# Validation of the simulations

This notebook reads the archives written by the validation runs (`validation/validationXX_*.py`, saved in `data/` as `validationXX_*.npz`) and the figure archives. Each section prints the numbers to check, and sections 1 to 7 draw a plot, saved to `validation/plots/`.

Sections:

1. **Exact flow** (`validation/validation01_exact_flow.py`). Euler simulator against RK4 and the closed form of Proposition 1, at small $d$, for $\tau \neq 1$ and both rank regimes.
2. **Step size, Figure 6** (`validation/validation02_stepsize_fig6.py`). Top-space distances with $\eta$, $\eta/2$, $\eta/4$.
3. **Step size, Figure 7** (`validation/validation03_stepsize_fig7.py`). Activation times of the singular modes with $\eta$, $\eta/2$, $\eta/4$.
4. **Step size, Figure 8** (`validation/validation04_stepsize_fig8.py`). Loss and mass below $x_s$ with $\eta$, $\eta/2$, $\eta/4$.
5. **Rotated teacher** (`validation/validation05_rotation.py`). Diagonal against rotated $C$, and factor against prediction projectors.
6. **Finite size** (`validation/validation06_dimension.py`). Convergence in $d$ to Theorems 4 and 6, for $\tau = 0.7, 1, 1.4$.
7. **Theory numerics** (no run). Densities of Theorem 4 with $\varepsilon$ halved and twice the quadrature nodes.
8. **Exact identities on the figure archives** (no run). Proposition 2 on the stored spectra, and factor against prediction projectors in Figures 2 and 5.

Convention for step-size checks: if a quantity $D$ has an error of first order in $\eta$, then $D(\eta) - D(\eta/2) \approx 2\,(D(\eta/2) - D(\eta/4))$, and the error of the figure run is about $2\,|D(\eta) - D(\eta/2)|$. The ratio printed as `order ratio` should be close to 2.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root (the folder containing src/).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import DATA_DIR, VALIDATION_PLOTS_DIR
from src.theory import SignalLaw, make_theory


def paper_density(x):
    """Same smooth non-uniform law as src.laws.paper_phi."""
    x = np.asarray(x, dtype=float)
    a, b = 0.35, 2.0
    u = (x - a) / (b - a)
    left_bump = 0.75 * np.exp(-0.5 * ((u - 0.30) / 0.13) ** 2)
    right_bump = 0.45 * np.exp(-0.5 * ((u - 0.70) / 0.18) ** 2)
    shoulder = 0.16 * (1.0 + 0.35 * np.cos(2.0 * np.pi * u))
    out = np.zeros_like(x)
    mask = (x >= a) & (x <= b)
    out[mask] = np.maximum(left_bump[mask] + right_bump[mask] + shoulder[mask], 0.0)
    return out


law = SignalLaw.from_pdf(paper_density, (0.35, 2.0), name="paper_two_bump")

VALIDATION_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    "U": "#2B6F9E",
    "V": "#009E8E",
    "C": "#202124",
    "accent": "#D95F02",
    "rho1": "#D1495B",
    "rho2": "#F28E2B",
    "rho3": "#1B9E77",
    "rho4": "#4E79A7",
    "rho5": "#B07AA1",
    "muted": "#7A7F87",
}
ETA_STYLES = {1: dict(ls="-", lw=1.8), 2: dict(ls="--", lw=1.4), 4: dict(ls=":", lw=1.6)}

mpl.rcParams.update(mpl.rcParamsDefault)
plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.family": "serif",
    "axes.unicode_minus": False,
    "figure.dpi": 120,
    "savefig.dpi": 240,
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.2,
})

figwidth = 10

In [ ]:
def load_npz(path):
    """Load a simulation archive (NumPy only)."""
    with np.load(path, allow_pickle=False) as data:
        out = {key: data[key] for key in data.files if not key.endswith("_json")}
        out["config"] = json.loads(str(data["config_json"]))
        out["dimensions"] = json.loads(str(data["dimensions_json"]))
    return out


def load_if_exists(name, script):
    """Load data/<name>.npz, or print the script to run and return None."""
    path = DATA_DIR / f"{name}.npz"
    if not path.exists():
        print(f"missing {path.name}: run `python {script}` first.")
        return None
    return load_npz(path)


def savefig(fig, name):
    path = VALIDATION_PLOTS_DIR / name
    fig.savefig(path)
    print(path)


def nanmax(x, axis=None):
    """np.nanmax without the warning on all-NaN slices (returns NaN there)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmax(x, axis=axis)


def nanmean(x, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(x, axis=axis)


def nanstd(x, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanstd(x, axis=axis)


def rho_color(index):
    return [COLORS["rho1"], COLORS["rho2"], COLORS["rho3"], COLORS["rho4"], COLORS["rho5"]][index % 5]


def rho_index(data, rho):
    """Index of the value of rho closest to `rho` (in log scale) in an archive."""
    rhos = np.asarray(data["rhos"], dtype=float).reshape(-1)
    return int(np.argmin(np.abs(np.log(rhos) - np.log(float(rho)))))


def time_indices(times_main, times_check):
    """Indices of `times_check` inside `times_main`, or None if some time is missing."""
    times_main = np.asarray(times_main, dtype=float)
    idx = np.searchsorted(times_main, np.asarray(times_check, dtype=float) - 1e-9)
    idx = np.clip(idx, 0, times_main.size - 1)
    if not np.allclose(times_main[idx], times_check, rtol=0.0, atol=1e-8):
        return None
    return idx


def theory_of(data, n_quad=600):
    """High-dimensional theory with the parameters stored in an archive (paper law)."""
    cfg = data["config"]
    rhos = np.asarray(cfg["rhos"], dtype=float).reshape(-1)
    return make_theory(
        law,
        kappa=float(cfg["kappa"]),
        tau=float(cfg["tau"]),
        kappa_star=float(cfg["kappa_star"]),
        rho=float(rhos[0]),
        n_quad=n_quad,
    )


def step_summary(label, d1, d2, d4=None):
    """Print max|D(eta) - D(eta/2)|, max|D(eta/2) - D(eta/4)| and their ratio (about 2 at first order)."""
    diff_12 = nanmax(np.abs(np.asarray(d1) - np.asarray(d2)))
    line = f"  {label}: max|eta - eta/2| = {diff_12:.2e}"
    if d4 is not None:
        diff_24 = nanmax(np.abs(np.asarray(d2) - np.asarray(d4)))
        ratio = diff_12 / diff_24 if diff_24 > 0 else np.nan
        line += f", max|eta/2 - eta/4| = {diff_24:.2e}, order ratio = {ratio:.2f}"
    print(line)
    return diff_12

## 1. Euler simulator against the exact flow

Small $d = 40$, both rank regimes, $\tau \in \{0.6, 1, 1.5\}$ and $\rho \in \{10^{-2}, 10^{-6}\}$, with the stepsize $\eta = 10^{-3}$.

What to expect:
- `closed vs RK4` validates both references: about $10^{-9}$ or less, wherever the closed form is evaluated (early times, where its linear system is well conditioned).
- `Euler vs RK4` is the discretization error of the simulator, of order $\eta$ times the number of growth rates crossed, so roughly $10^{-3}$ to $10^{-2}$ in relative Frobenius norm, largest during the transitions.
- The loss and projector distances should agree to the same order. The RK4 imbalance error stays near round-off, while Euler drifts slowly.

In [ ]:
V1_SCRIPT = "validation/validation01_exact_flow.py"
V1_REGIMES = ("under", "over")
V1_TAUS = ("0p6", "1", "1p5")


fig, axes = plt.subplots(1, 2, figsize=(figwidth, 0.36 * figwidth), sharey=True, layout="constrained")
found_any = False
for ax, regime in zip(axes, V1_REGIMES):
    print(f"{regime}parameterized regime")
    for j, tau in enumerate(V1_TAUS):
        data = load_if_exists(f"validation01_exact_flow_{regime}_tau{tau}", V1_SCRIPT)
        if data is None:
            continue
        found_any = True
        times = np.asarray(data["times"], dtype=float)
        positive = times > 0
        for r, rho in enumerate(np.asarray(data["rhos"], dtype=float)):
            z_euler = nanmax(data["z_error_euler"][r], axis=-1)
            z_closed = nanmax(data["z_error_closed"][r], axis=-1)
            loss0 = float(nanmean(data["loss"][r, 0]))
            loss_gap = nanmax(np.abs(data["loss"][r] - data["rk4_loss"][r])) / loss0
            right_gap = nanmax(np.abs(data["prediction_right_delta"][r] - data["rk4_prediction_right_delta"][r]))
            left_gap = nanmax(np.abs(data["prediction_left_delta"][r] - data["rk4_prediction_left_delta"][r]))
            evaluated = np.mean(np.isfinite(data["z_error_closed"][r][positive]))
            print(
                f"  tau = {tau.replace('p', '.')}, rho = {rho:.0e}: "
                f"Z Euler vs RK4 {nanmax(z_euler):.1e}, "
                f"closed vs RK4 {nanmax(z_closed):.1e} (on {100 * evaluated:.0f}% of times), "
                f"loss {loss_gap:.1e}, Delta_L {left_gap:.1e}, Delta_R {right_gap:.1e}, "
                f"imbalance Euler {nanmax(data['imbalance_error'][r]):.1e} / RK4 {nanmax(data['rk4_imbalance_error'][r]):.1e}"
            )
            style = dict(color=rho_color(j), ls="-" if r == 0 else "--", lw=1.5)
            label = rf"$\tau={tau.replace('p', '.')}$, $\rho=10^{{{int(round(np.log10(rho)))}}}$"
            ax.plot(times[positive], z_euler[positive], label=label, **style)
            ax.plot(times[positive], z_closed[positive], color=rho_color(j), ls=":", lw=1.2)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$t$")
    ax.set_title(f"{regime}parameterized")
axes[0].set_ylabel(r"$\|\mathbf{Z} - \mathbf{Z}_{\mathrm{RK4}}\|_F / \|\mathbf{Z}_{\mathrm{RK4}}\|_F$")
if found_any:
    axes[0].legend(frameon=True, fontsize=8, ncol=2, loc="lower right")
    fig.text(0.5, -0.02, "solid / dashed: Euler simulator; dotted: closed form (where evaluated)", ha="center", fontsize=9)
    savefig(fig, "validation01_exact_flow.pdf")
    plt.show()
else:
    plt.close(fig)

## 2. Step size, Figure 6

Repetition 0 of both Figure 6 regimes at $\rho = 10^{-5}$ and $10^{-25}$, with $\eta$, $\eta/2$ and $\eta/4$. The distance is the one of the figure: the average of the left and right top-$k$ distances of $\mathbf{Z}$, scaled by $k/d$.

Differences are measured in $\log_{10}$, since the distances span many orders of magnitude (only values above $10^{-12}$ are compared). The section also prints the gap between the factor projectors ($\mathbf{U}$, $\mathbf{V}$) and the prediction projectors ($\mathbf{Z}$). They coincide under the exact flow, so this gap is a second measure of the Euler error and should shrink with $\eta$.

In [ ]:
V2_SCRIPT = "validation/validation02_stepsize_fig6.py"
V2_FLOOR = 1e-12


def fig6_top_distance(data, rho, rep=0):
    """Figure 6 distance: mean of left and right top-k distances of Z, scaled by k/d."""
    r = rho_index(data, rho)
    d = int(data["dimensions"]["d"])
    k = int(np.floor(float(np.asarray(data["top_eig_fracs"]).reshape(-1)[0]) * d))
    left = np.asarray(data["prediction_top_left_delta"], dtype=float)[r, :, 0, rep]
    right = np.asarray(data["prediction_top_right_delta"], dtype=float)[r, :, 0, rep]
    return 0.5 * (left + right) * k / d


def fig6_factor_gap(data, rho, rep=0):
    """max over the save times after the first positive one of |factor - prediction| top-k distances (left and right)."""
    r = rho_index(data, rho)
    times = np.asarray(data["times"], dtype=float)
    later = times > times[1]
    gaps = [
        np.abs(np.asarray(data[a], dtype=float)[r, later, 0, rep] - np.asarray(data[b], dtype=float)[r, later, 0, rep])
        for a, b in (("top_delta_u", "prediction_top_left_delta"), ("top_delta_v", "prediction_top_right_delta"))
    ]
    return nanmax(np.concatenate(gaps))


def log_distance(y):
    y = np.asarray(y, dtype=float)
    return np.where(y > V2_FLOOR, np.log10(np.maximum(y, V2_FLOOR)), np.nan)


fig, axes = plt.subplots(1, 2, figsize=(figwidth, 0.36 * figwidth), layout="constrained")
found_any = False
for ax, regime in zip(axes, ("continuous", "gapped")):
    main = load_if_exists(f"figure06_{regime}_rates", "figures/figure06.py")
    checks = {k: load_if_exists(f"validation02_stepsize_fig6_{regime}_eta_over{k}", V2_SCRIPT) for k in (2, 4)}
    if main is None or any(c is None for c in checks.values()):
        ax.set_axis_off()
        continue
    idx = time_indices(main["times"], checks[2]["times"])
    if idx is None or not np.allclose(checks[2]["times"], checks[4]["times"]):
        print(f"{regime}: save times do not match the figure run (was a --quick or --t-max option used?).")
        ax.set_axis_off()
        continue
    found_any = True
    times = np.asarray(checks[2]["times"], dtype=float)
    print(f"{regime} spectrum")
    for j, rho in enumerate(np.asarray(checks[2]["rhos"], dtype=float)):
        curves = {
            1: fig6_top_distance(main, rho)[idx],
            2: fig6_top_distance(checks[2], rho),
            4: fig6_top_distance(checks[4], rho),
        }
        step_summary(f"rho = {rho:.0e}, log10 distance", *(log_distance(curves[k]) for k in (1, 2, 4)))
        print(
            "    factor vs prediction projectors: "
            + ", ".join(f"{name} {fig6_factor_gap(data, rho):.1e}" for name, data in (("eta", main), ("eta/2", checks[2]), ("eta/4", checks[4])))
        )
        for k, curve in curves.items():
            label = rf"$\rho=10^{{{int(round(np.log10(rho)))}}}$" if k == 1 else None
            ax.plot(times[times > 0], curve[times > 0], color=rho_color(j), label=label, **ETA_STYLES[k])
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$t$")
    ax.set_title(f"{regime} spectrum")
    ax.legend(frameon=True, loc="lower left")
if found_any:
    axes[0].set_ylabel(r"top-$k$ distance ($\eta$ solid, $\eta/2$ dashed, $\eta/4$ dotted)")
    savefig(fig, "validation02_stepsize_fig6.pdf")
    plt.show()
else:
    plt.close(fig)

## 3. Step size, Figure 7

Both Figure 7 panels ($d = 24$ and $96$, the four values of $\rho$), rerun with $\eta$, $\eta/2$ and $\eta/4$ with the singular values of $\mathbf{Z}$ stored. The activation time of mode $i$ is the first time at which $\sigma_i(\mathbf{Z}(t))$ reaches half of $\sigma_i(\mathbf{C})$, linearly interpolated between save times, and is reported in the variable $s = t/\log(1/\rho)$ of the figure. Modes that do not activate before $t_{\max} = 110$ are ignored.

What to expect: the shifts between stepsizes are of order $\eta\,\sigma_i/2$ relative to the activation time, so about $10^{-3}$ relative. The last column compares with the $\rho \to 0$ prediction $s_i = 1/(2\sigma_i)$ of Theorem 11, which is approached as $\rho$ decreases.

In [ ]:
V3_SCRIPT = "validation/validation03_stepsize_fig7.py"


def eta_tag(factor):
    """Same file-name tags as src.runs.eta_tag."""
    return "eta" if factor == 1 else f"eta_over{factor}"


def activation_times(data, rho_idx):
    """Time at which sigma_i(Z) first reaches sigma_i(C) / 2, for each selected mode (NaN if never)."""
    times = np.asarray(data["times"], dtype=float)
    sv = np.asarray(data["singular_values_z"], dtype=float)[rho_idx, :, 0, :]
    teacher = np.asarray(data["teacher_singular_values"], dtype=float)[0]
    rank = int(data["dimensions"]["selected_rank"])
    out = np.full(rank, np.nan)
    for i in range(rank):
        target = 0.5 * teacher[i]
        above = np.flatnonzero(sv[:, i] >= target)
        if above.size == 0 or above[0] == 0:
            continue
        k = above[0]
        y0, y1 = sv[k - 1, i], sv[k, i]
        out[i] = times[k - 1] + (target - y0) * (times[k] - times[k - 1]) / (y1 - y0)
    return out, teacher[:rank]


fig, axes = plt.subplots(1, 2, figsize=(figwidth, 0.36 * figwidth), sharey=True, layout="constrained")
found_any = False
for ax, d in zip(axes, (24, 96)):
    runs = {k: load_if_exists(f"validation03_stepsize_fig7_d{d}_{eta_tag(k)}", V3_SCRIPT) for k in (1, 2, 4)}
    if any(run is None for run in runs.values()):
        ax.set_axis_off()
        continue
    found_any = True
    print(f"d = {d}")
    for j, rho in enumerate(np.asarray(runs[1]["rhos"], dtype=float)):
        log_inv = np.log(1.0 / rho)
        s = {}
        for k, run in runs.items():
            t_act, sigma = activation_times(run, rho_index(run, rho))
            s[k] = t_act / log_inv
        rel = np.abs(s[1] - s[2]) / s[2]
        theory_gap = nanmax(np.abs(s[4] - 1.0 / (2.0 * sigma)))
        print(
            f"  rho = {rho:.0e}: {np.sum(np.isfinite(s[4]))}/{sigma.size} modes active, "
            f"max |s(eta) - s(eta/2)| = {nanmax(np.abs(s[1] - s[2])):.1e} (relative {nanmax(rel):.1e}), "
            f"max |s(eta/2) - s(eta/4)| = {nanmax(np.abs(s[2] - s[4])):.1e}, "
            f"max |s(eta/4) - 1/(2 sigma)| = {theory_gap:.1e}"
        )
        ax.plot(sigma, s[1] - s[4], "o-", ms=3, lw=1.0, color=rho_color(j), label=rf"$\rho=10^{{{int(round(np.log10(rho)))}}}$")
    ax.axhline(0.0, color=COLORS["muted"], lw=0.8)
    ax.set_xlabel(r"$\sigma_i(\mathbf{C})$")
    ax.set_title(rf"$d={d}$")
if found_any:
    axes[0].set_ylabel(r"$s_i(\eta) - s_i(\eta/4)$")
    axes[0].legend(frameon=True)
    savefig(fig, "validation03_stepsize_fig7.pdf")
    plt.show()
else:
    plt.close(fig)

## 4. Step size, Figure 8

The first two repetitions of Figure 8, rerun with $\eta/2$ and $\eta/4$ (same seeds and save times), are compared with the same repetitions of `figure08_spectral_measures.npz` on the window $s = t/\log(1/\rho) \in [0.1, 0.7]$ of the lower panels. Two quantities are compared: the loss $\|\mathbf{Z}-\mathbf{C}\|_F^2/\min(p,d)$ (relative to its maximum), and the fraction of singular values of $\mathbf{Z}$ below $x_s$ (pooled over the two repetitions).

In [ ]:
V4_SCRIPT = "validation/validation04_stepsize_fig8.py"
V4_WINDOW = (0.1, 0.7)

main7 = load_if_exists("figure08_spectral_measures", "figures/figure08.py")
checks7 = {k: load_if_exists(f"validation04_stepsize_fig8_eta_over{k}", V4_SCRIPT) for k in (2, 4)}
if main7 is not None and all(c is not None for c in checks7.values()):
    rho7 = float(np.asarray(main7["rhos"], dtype=float).reshape(-1)[0])
    n_check = int(np.asarray(checks7[2]["loss"]).shape[-1])
    times7 = np.asarray(checks7[2]["times"], dtype=float)
    idx7 = time_indices(main7["times"], times7)
    if idx7 is None or not np.allclose(times7, checks7[4]["times"]):
        print("The save times do not match the figure run (was a --quick or --t-max option used?).")
    else:
        theory7 = theory_of(main7)
        floor7 = max(float(theory7.paper_x_star), float(theory7.law.support[0]))
        s7 = times7 / np.log(1.0 / rho7)
        window = (s7 >= V4_WINDOW[0]) & (s7 <= V4_WINDOW[1])
        threshold7 = np.maximum(1.0 / (2.0 * s7[window]), floor7)

        def fig8_curves(data, index=None):
            """Mean loss over the paired repetitions, and pooled fraction of singular values below x_s."""
            loss = np.asarray(data["loss"], dtype=float)[0][:, :n_check]
            sv = np.asarray(data["singular_values_z"], dtype=float)[0][:, :n_check, :]
            if index is not None:
                loss, sv = loss[index], sv[index]
            loss, sv = loss[window], sv[window]
            finite = np.isfinite(sv)
            below = np.sum(finite & (np.abs(sv) < threshold7[:, None, None]), axis=(1, 2))
            return nanmean(loss, axis=-1), below / np.maximum(np.sum(finite, axis=(1, 2)), 1)

        loss7, mass7 = {}, {}
        loss7[1], mass7[1] = fig8_curves(main7, idx7)
        for k in (2, 4):
            loss7[k], mass7[k] = fig8_curves(checks7[k])
        scale = nanmax(loss7[1])
        print(f"Figure 8 step-size check on {n_check} paired repetitions, s in {V4_WINDOW}:")
        loss_shift = step_summary("loss / max loss", loss7[1] / scale, loss7[2] / scale, loss7[4] / scale)
        mass_shift = step_summary("mass below x_s", mass7[1], mass7[2], mass7[4])
        value = max(loss_shift, mass_shift)
        mantissa, exponent = f"{value:.0e}".split("e")
        print(f"\nStep-size error (eta -> eta/2): {value:.2e}, i.e. ${mantissa} \\cdot 10^{{{int(exponent)}}}$")

        fig, axes = plt.subplots(1, 2, figsize=(figwidth, 0.34 * figwidth), layout="constrained")
        s_win = s7[window]
        for ax, curves, title in ((axes[0], {k: v / scale for k, v in loss7.items()}, r"Loss / max loss"), (axes[1], mass7, r"Mass below $x_s$")):
            ax.plot(s_win, np.abs(curves[1] - curves[4]), color=COLORS["accent"], lw=1.6, label=r"$|D(\eta) - D(\eta/4)|$")
            ax.plot(s_win, np.abs(curves[2] - curves[4]), color=COLORS["C"], lw=1.3, ls="--", label=r"$|D(\eta/2) - D(\eta/4)|$")
            ax.set_yscale("log")
            ax.set_xlabel(r"$s=t/\log(1/\rho)$")
            ax.set_title(title)
        axes[0].legend(frameon=True)
        savefig(fig, "validation04_stepsize_fig8.pdf")
        plt.show()

## 5. Rotated teacher, and factor against prediction projectors

$d = 200$, both rank regimes, $\rho = 10^{-5}$, 4 repetitions. The teacher is either diagonal (as in all figures) or has random singular vectors, with the same singular values and the same initialization $\mathbf{R}$. The dynamics are equal in law, so the averaged curves must agree up to finite-size fluctuations. For each observable the cell prints the largest gap between the two means, and the same gap divided by its standard error (`z`). With 4 repetitions and about 150 save times, values of `z` up to about 3 are normal. Times where both distances are below $10^{-12}$ (round-off level) are ignored for `z`.

Second check: the range projectors of $\mathbf{U}$ and $\mathbf{V}$ (rank found with a relative SVD threshold of $10^{-7}$) against the singular projectors of $\mathbf{Z}$ (exact rank $\min(m, m^*)$). By Proposition 2 they coincide on the left side in both regimes, and on the right side in the underparameterized regime. A gap far above $10^{-10}$ would mean that the SVD threshold misses or adds a direction.

In [ ]:
V5_SCRIPT = "validation/validation05_rotation.py"


Z_FLOOR = 1e-12


def mean_gap(a, b, floor=Z_FLOOR):
    """Largest gap between the repetition means of a and b, raw and in units of its standard error.

    Times where both means are below `floor` (round-off level) are ignored for the z score.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n_a, n_b = a.shape[-1], b.shape[-1]
    mean_a, mean_b = nanmean(a, axis=-1), nanmean(b, axis=-1)
    gap = np.abs(mean_a - mean_b)
    err = np.sqrt(nanstd(a, axis=-1) ** 2 / n_a + nanstd(b, axis=-1) ** 2 / n_b)
    resolved = (err > 0) & (np.maximum(np.abs(mean_a), np.abs(mean_b)) > floor)
    z = np.where(resolved, gap / np.where(err > 0, err, 1.0), np.nan)
    return nanmax(gap), nanmax(z)


def factor_prediction_gap(data, side):
    """max over t > 0, rho and repetitions of |range distance of U or V - singular-space distance of Z|."""
    factor_key, prediction_key = {"left": ("range_delta_u", "prediction_left_delta"), "right": ("range_delta_v", "prediction_right_delta")}[side]
    positive = np.asarray(data["times"], dtype=float) > 0
    return nanmax(np.abs(np.asarray(data[factor_key])[:, positive] - np.asarray(data[prediction_key])[:, positive]))


fig, axes = plt.subplots(1, 2, figsize=(figwidth, 0.36 * figwidth), layout="constrained")
found_any = False
for ax, regime in zip(axes, ("under", "over")):
    runs = {basis: load_if_exists(f"validation05_rotation_{regime}_{basis}", V5_SCRIPT) for basis in ("diagonal", "rotated")}
    if any(run is None for run in runs.values()):
        ax.set_axis_off()
        continue
    found_any = True
    diag, rot = runs["diagonal"], runs["rotated"]
    times = np.asarray(diag["times"], dtype=float)
    positive = times > 0
    print(f"{regime}parameterized regime")
    keys = ["loss", "prediction_right_delta"] + (["prediction_left_delta"] if regime == "under" else [])
    for key in keys:
        gap, z = mean_gap(diag[key][0][positive], rot[key][0][positive])
        print(f"  {key}: max gap of the means {gap:.2e}, max z = {z:.1f}")
    for j, frac in enumerate(np.asarray(diag["top_eig_fracs"], dtype=float)):
        for side in ("left", "right"):
            key = f"prediction_top_{side}_delta"
            gap, z = mean_gap(diag[key][0][positive, j], rot[key][0][positive, j])
            print(f"  {key} (k = {frac:g} d): max gap of the means {gap:.2e}, max z = {z:.1f}")
    for name, run in runs.items():
        sides = ("left", "right") if regime == "under" else ("left",)
        gaps = ", ".join(f"{side} {factor_prediction_gap(run, side):.1e}" for side in sides)
        print(f"  factor vs prediction projectors ({name} teacher): {gaps}")

    for key, color in (("prediction_right_delta", COLORS["V"]), ("prediction_left_delta", COLORS["U"])):
        if key == "prediction_left_delta" and regime != "under":
            continue
        side = "R" if "right" in key else "L"
        for run, ls, basis in ((diag, "-", "diagonal"), (rot, "--", "rotated")):
            values = np.asarray(run[key], dtype=float)[0][positive]
            mean, std = nanmean(values, axis=-1), nanstd(values, axis=-1)
            ax.plot(times[positive], mean, color=color, ls=ls, lw=1.5, label=rf"$\Delta^\mathsf{{{side}}}(t)^2$, {basis}")
            ax.fill_between(times[positive], np.maximum(mean - std, 1e-300), mean + std, color=color, alpha=0.12, linewidth=0)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$t$")
    ax.set_title(f"{regime}parameterized")
    ax.legend(frameon=True, fontsize=8, loc="lower left")
if found_any:
    savefig(fig, "validation05_rotation.pdf")
    plt.show()
else:
    plt.close(fig)

## 6. Finite size: convergence to the high-dimensional theory

Figure 4 setting ($\kappa = 0.55$, $\kappa^* = 0.9$, $\rho = 0.1$) at $t \in \{0.5, 1.2, 3.5, 8\}$, for $d = 100$ to $800$ and $\tau \in \{0.7, 1, 1.4\}$. The singular values of $\mathbf{C}$ are deterministic quantiles of $\phi_C$, so only the initialization $\mathbf{R}$ is random, and every $d$ uses 16 repetitions. Two comparisons, both at the same $\rho$ (no $\rho \to 0$ limit):

- the Stieltjes transforms of $\mathbf{U}\mathbf{U}^\top$ and $\mathbf{V}\mathbf{V}^\top$ at $w = -0.5$ and $w = 1 + 0.5i$ (Theorem 4), computed exactly from the stored eigenvalues, so with no smoothing parameter;
- the range-projector distances $\Delta^\mathsf{L}(t)^2$ and $\Delta^\mathsf{R}(t)^2$ (Theorem 6).

What to expect: errors that decrease roughly like $1/d$ (dotted reference line), for all three values of $\tau$. The paper figures only use $\tau = 1$, so this is the only test of the $\tau$-dependent formulas against simulations.

In [ ]:
V6_SCRIPT = "validation/validation06_dimension.py"
V6_DIMS = (100, 200, 400, 800)
V6_TAUS = ("0p7", "1", "1p4")
V6_W = (-0.5 + 0.0j, 1.0 + 0.5j)
V6_BRANCHES = (("U", "eigvals_u", "prediction_left_delta", "L"), ("V", "eigvals_v", "prediction_right_delta", "R"))

fig, axes = plt.subplots(1, 2, figsize=(figwidth, 0.36 * figwidth), sharex=True, layout="constrained")
found_any = False
for j, tau in enumerate(V6_TAUS):
    runs = {d: load_if_exists(f"validation06_dimension_tau{tau}_d{d}", V6_SCRIPT) for d in V6_DIMS}
    runs = {d: run for d, run in runs.items() if run is not None}
    if not runs:
        continue
    found_any = True
    first = next(iter(runs.values()))
    theory = theory_of(first)
    times = np.asarray(first["times"], dtype=float)
    positive = np.flatnonzero(times > 0)
    # The theory does not depend on d: compute it once per tau.
    m_theory = {(b, i, w): theory.stieltjes(b, float(times[i]), complex(w)) for b, *_ in V6_BRANCHES for i in positive for w in V6_W}
    delta_theory = {(b, i): theory.projector_delta(b, float(times[i])) for b, *_ in V6_BRANCHES for i in positive}

    print(f"tau = {tau.replace('p', '.')}")
    dims, stieltjes_err, projector_err = [], {b: [] for b, *_ in V6_BRANCHES}, {b: [] for b, *_ in V6_BRANCHES}
    for d, run in runs.items():
        dims.append(d)
        line = f"  d = {d:4d} ({run['loss'].shape[-1]} reps):"
        for branch, eig_key, delta_key, side in V6_BRANCHES:
            err_m, err_delta = 0.0, 0.0
            for i in positive:
                eigs = np.asarray(run[eig_key], dtype=float)[0, i]  # (reps, n)
                for w in V6_W:
                    m_emp = np.mean(1.0 / (eigs - w))
                    err_m = max(err_m, abs(m_emp - m_theory[(branch, i, w)]))
                delta_emp = float(nanmean(np.asarray(run[delta_key], dtype=float)[0, i]))
                err_delta = max(err_delta, abs(delta_emp - delta_theory[(branch, i)]))
            stieltjes_err[branch].append(err_m)
            projector_err[branch].append(err_delta)
            line += f" |m_{branch} - theory| {err_m:.1e}, |Delta_{side} - theory| {err_delta:.1e};"
        print(line.rstrip(";"))
    color = rho_color(j)
    label_tau = rf"$\tau={tau.replace('p', '.')}$"
    for branch, marker in (("U", "o"), ("V", "s")):
        side = "L" if branch == "U" else "R"
        ls = "-" if branch == "U" else "--"
        axes[0].plot(dims, stieltjes_err[branch], marker=marker, ms=4, color=color, ls=ls, label=f"{label_tau}, $m^{branch}$")
        axes[1].plot(dims, projector_err[branch], marker=marker, ms=4, color=color, ls=ls, label=rf"{label_tau}, $\Delta^\mathsf{{{side}}}$")

if found_any:
    ref = np.array(V6_DIMS, dtype=float)
    for ax, title in zip(axes, ("Stieltjes transforms", "Range-projector distances")):
        ax.set_xscale("log")
        ax.set_yscale("log")
        ylim = ax.get_ylim()
        # 1/d reference line through the middle of the panel.
        ax.plot(ref, np.sqrt(ylim[0] * ylim[1]) * np.sqrt(ref[0] * ref[-1]) / ref, color=COLORS["muted"], ls=":", lw=1.0, label=r"$\propto 1/d$")
        ax.set_ylim(ylim)
        ax.set_xlabel(r"$d$")
        ax.set_title(title)
        ax.legend(frameon=True, fontsize=7, ncol=2)
    axes[0].set_ylabel(r"max error over $t$ (and $w$)")
    savefig(fig, "validation06_dimension.pdf")
    plt.show()
else:
    plt.close(fig)

## 7. Theory numerics: smoothing and quadrature

No simulation needed. The density of $\mu_t^U$ (Theorem 4, Figure 4 setting) is computed from the Stieltjes transform at $w = x + i\varepsilon$, with a Gauss–Legendre quadrature of the law $\phi_C$. This cell recomputes it with $\varepsilon$ halved and with twice the quadrature nodes, without clipping negative values. It prints:

- the largest change of the density, relative to its maximum;
- the most negative value before clipping (should be about 0, otherwise the solver picked a wrong branch);
- the largest residual of the self-consistent equation;
- the total mass, continuous part plus atom at zero (should be close to 1, up to about $\varepsilon$ and the grid range).

What to expect: changes with $\varepsilon$ are visible where the density is sharp (early times, near $x = 0$) and small elsewhere. Changes with the number of nodes should be negligible. This takes about a minute.

In [ ]:
from scipy.integrate import trapezoid

V7_TIMES = (0.5, 1.2, 3.5, 8.0, 50.0)
V7_GRID = np.linspace(1e-3, 2.5, 600)
V7_SETTINGS = {
    "reference": dict(eta=5e-3, n_quad=600, ls="-"),
    "epsilon / 2": dict(eta=2.5e-3, n_quad=600, ls="--"),
    "2 x nodes": dict(eta=5e-3, n_quad=1200, ls=":"),
}
v7_theories = {n: make_theory(law, kappa=0.55, tau=1.0, kappa_star=0.9, rho=0.1, n_quad=n) for n in (600, 1200)}

v7 = {}
for name, setting in V7_SETTINGS.items():
    theory = v7_theories[setting["n_quad"]]
    v7[name] = [theory.density("U", V7_GRID, t, eta=setting["eta"], clip_negative=False) for t in V7_TIMES]

fig, axes = plt.subplots(1, len(V7_TIMES), figsize=(figwidth, 0.26 * figwidth), layout="constrained")
for i, (t, ax) in enumerate(zip(V7_TIMES, axes)):
    ref = v7["reference"][i]["density_unnormalized"]
    scale = nanmax(np.abs(ref))
    print(f"t = {t:g}")
    for name, results in v7.items():
        out = results[i]
        dens = out["density_unnormalized"]
        mass = trapezoid(dens, V7_GRID) + float(out["atom_zero"])
        change = nanmax(np.abs(dens - ref)) / scale
        print(
            f"  {name:12s}: change {change:.1e}, min density {np.min(dens):.1e}, "
            f"max residual {nanmax(out['residual']):.1e}, total mass {mass:.4f}"
        )
        ax.plot(V7_GRID, dens, lw=1.3, ls=V7_SETTINGS[name]["ls"], label=name)
    ax.set_title(rf"$t={t:g}$")
    ax.set_xlabel(r"$x$")
axes[0].set_ylabel(r"density of $\mu_t^U$")
axes[-1].legend(frameon=True, fontsize=8)
savefig(fig, "validation07_theory_numerics.pdf")
plt.show()

## 8. Exact identities on the figure archives

No simulation needed. Under the exact flow, Proposition 2 gives, with $r = \min(m, m^*)$:

- `shift`: $\lambda_i^V = \lambda_i^U + 2\rho$ for $i < r$ (imbalance conservation);
- `frozen`: $\lambda_i^V = 2\rho$ for $r \le i < m$ (overparameterized case);
- `zeros U`, `zeros V`: $\lambda_i^U = 0$ for $i \ge r$ and $\lambda_i^V = 0$ for $i \ge m$;
- `pushforward`: $\sigma_i(\mathbf{Z}) = \sqrt{\lambda_i^U(\lambda_i^U + 2\rho)}$ for $i < r$.

The cell measures these on every stored spectrum (Figures 3, 4 and 8), as absolute errors and relative to the largest eigenvalue. Explicit Euler conserves the imbalance only up to $O(\eta)$ over a run, so `shift`, `frozen` and `pushforward` show a small drift. The zeros are exact up to round-off, since Euler keeps $\mathbf{U}$ in the range of $\mathbf{C}$. Round-off is about $10^{-7}$ relative in float32 (Figure 4) and about $10^{-15}$ in float64.

The cell also compares the range projectors of $\mathbf{U}$, $\mathbf{V}$ with the singular projectors of $\mathbf{Z}$ in the Figure 2 and 5 archives, as in section 5. The same comparison for Figure 6 is in section 2.

In [ ]:
V8_SPECTRA = {
    "Figure 3, under": ("figure03_convergence_under", "figures/figure03.py"),
    "Figure 3, over": ("figure03_convergence_over", "figures/figure03.py"),
    "Figure 4": ("figure04_density_comparison", "figures/figure04.py"),
    "Figure 8": ("figure08_spectral_measures", "figures/figure08.py"),
}
V8_PROJECTORS = {
    "Figure 2": ("figure02_visual_abstract", "figures/figure02.py"),
    "Figure 5, under": ("figure05_under", "figures/figure05.py"),
    "Figure 5, over l=0": ("figure05_over_exponent_l0", "figures/figure05.py"),
    "Figure 5, over l=1": ("figure05_over_exponent_l1", "figures/figure05.py"),
}


def svd_identities(data):
    """Largest violations of the identities of Proposition 2 over all stored times > 0 and repetitions."""
    dims = data["dimensions"]
    r, m = int(dims["selected_rank"]), int(dims["m"])
    positive = np.asarray(data["times"], dtype=float) > 0
    out = {}
    for k, rho in enumerate(np.asarray(data["rhos"], dtype=float)):
        lam_u = np.asarray(data["eigvals_u"], dtype=float)[k, positive]
        lam_v = np.asarray(data["eigvals_v"], dtype=float)[k, positive]
        sv = np.asarray(data["singular_values_z"], dtype=float)[k, positive]
        scale = nanmax(lam_v)
        checks = {
            "shift": nanmax(np.abs(lam_v[..., :r] - lam_u[..., :r] - 2.0 * rho)),
            "frozen": nanmax(np.abs(lam_v[..., r:m] - 2.0 * rho)) if m > r else np.nan,
            "zeros U": nanmax(np.abs(lam_u[..., r:])) if lam_u.shape[-1] > r else np.nan,
            "zeros V": nanmax(np.abs(lam_v[..., m:])) if lam_v.shape[-1] > m else np.nan,
            "pushforward": nanmax(np.abs(sv[..., :r] - np.sqrt(np.maximum(lam_u[..., :r] * (lam_u[..., :r] + 2.0 * rho), 0.0)))),
        }
        out[rho] = (checks, scale)
    return out



print("Proposition 2 on the stored spectra (absolute error, then relative to the largest eigenvalue)")
for label, (name, script) in V8_SPECTRA.items():
    data = load_if_exists(name, script)
    if data is None:
        continue
    print(f"{label} ({data['config'].get('dtype')}):")
    for rho, (checks, scale) in svd_identities(data).items():
        parts = ", ".join(f"{key} {value:.1e} ({value / scale:.1e})" for key, value in checks.items() if np.isfinite(value))
        print(f"  rho = {rho:.0e}: {parts}")

print("\nRange projectors of U, V against singular projectors of Z")
for label, (name, script) in V8_PROJECTORS.items():
    data = load_if_exists(name, script)
    if data is None:
        continue
    dims = data["dimensions"]
    sides = ("left", "right") if int(dims["m"]) <= int(dims["m_star"]) else ("left",)
    gaps = ", ".join(f"{side} {factor_prediction_gap(data, side):.1e}" for side in sides)
    print(f"  {label}: {gaps}")